# Day 8(M2 Day04) 실습 — Multi-Tool Agent 구현

**목표**: `create_agent`로 도구 사용을 자동화하는 에이전트를 만들고, 다단계 복합 질문을 처리한다.
**구성**: Part 1 에이전트 기초(+오류 대응) → Part 2 다단계 복합 질문(+M1·M2 도구 적용) → Part 3 Multi-Tool Agent(+M1~M2 통합: 면접 코치 에이전트화)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키를 확인한다.

## 노트북 사용법

| 표시 | 뜻 |
| --- | --- |
| `# TODO` | **필수 실습** — 직접 코드를 채운다 |
| `# (관찰)` | 주어진 코드를 실행하고 출력만 읽는다 |

오늘의 필수 실습은 네 가지다.

1. `create_agent`로 도구를 등록한 에이전트 만들기
2. `messages` 형식으로 실행하고 최종 답 꺼내기
3. 다단계 복합 질문을 에이전트에 맡기기
4. 방어 로직을 앞뒤에 두고 에이전트 감싸기

> **참고:** 막히면 바로 정답을 열지 말고 앞 절에서 쓴 코드를 되짚는다 — 같은 패턴이 반복된다.

## 0. 환경 준비

In [1]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

### 0-보충. 프레임워크 없이 손으로 짜는 ReAct 루프

아래 Part 1은 `create_agent`가 도구 호출·반복·최종 답까지 전부 대신 처리한다. 그 전에 `create_agent`가 정확히 무엇을 대신해주는지 체감하기 위해, 같은 일을 **직접 손으로** 만들어본다 — Thought(생각) → Action(행동) → Observation(관찰)을 반복하다가 Answer(답)가 나오면 멈추는 루프다.

M1 Day02의 ReAct는 형식만 연습했다(Observation을 모델이 스스로 지어냈다). 여기서는 **실제로 도구를 실행**해서 진짜 Observation을 만든다.

In [3]:
# 도구
def get_weather(city: str) -> str:
    return f"{city}의 날씨는 맑음, 23도"

def calculate(a: float, b: float, op: str) -> str:
    table = {"+": a + b, "-": a - b, "*": a * b, "/": a / b if b else "0으로 나눌 수 없음"}
    return str(table.get(op, "알 수 없는 연산"))

In [9]:
# 프롬프트
react_system_prompt = """너는 아래 형식으로만 대답하는 에이전트이다. 도구가 필요하면 Action 까지만 쓰고 멈춘다.

    Thought: 지금 무엇을 해야 하는지 생각한다.
    Action: get_weather(city="도시명") 또는 calculate
    Objservation: 도구 실행결과가 여기에 채워진다
    필요하면 Thought에서 Action, Observation을 여러번 반복한다.
    ...
    Thought: 이제 답을 알겠다.
    Answer: 최종 답

"""

In [10]:
def show_history(history, step):
    print(f'--- step : {step} history')
    print(f'질문 : {history.split("질문:", 1)[1]}')

In [ ]:
import re

# react 구현
def run_react(question):
    history = f'{react_system_prompt}\n\n질문: {question}\n'

    output = llm.invoke(history).content
    print(output)
    # react 반복을 구현
    #1. 도구 이름 추출하기

    m = re.search(r'Action:\s*get_weather\(city="(.+?)"\)', output)
    m2 = re.search(r'Action:\s*calculate\(a=([\d.]+),\s*b=([\d.]+),\s*op="(.+?)"\)', output)

    if m:
        observation = get_weather(m.group(1))
    elif m2:
        observation = calculate(float(m2.group(1)), float(m2.group(2)), m2.group(2))
    elif "Answer:" in output: 
        return output.split("Answer:")[-1].strip()
    else:
        return "파싱실패: " + output

In [ ]:
run_react('서울 날씨의 현재 온도에 10도가 올라가면 몇 도가 되나요?')

Thought: 서울의 현재 온도를 알아야 한다.  
Action: get_weather(city="서울")  
Observation: 도구 실행결과가 여기에 채워진다.  
...  
Thought: 현재 온도를 알게 되었다.  
Action: calculate  
Observation: 도구 실행결과가 여기에 채워진다.  
...  
Thought: 이제 답을 알겠다.  
Answer: 최종 답  


## Part 1. create_agent 기초

완성 코드를 직접 쳐서 에이전트를 만들고 실행 과정을 관찰한다.

### 1-1. 도구 준비 + 에이전트 생성

In [18]:
# 도구
@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    return f"{city}의 날씨는 맑음, 23도"

@tool
def calculate(a: float, b: float, op: str) -> str:
    """두 수 a, b를 op(+,-,*,/)로 계산한다"""
    table = {"+": a + b, "-": a - b, "*": a * b, "/": a / b if b else "0으로 나눌 수 없음"}
    return str(table.get(op, "알 수 없는 연산"))

In [19]:
# TODO: get_weather·calculate를 tools로 넘겨 create_agent로 에이전트를 만드세요
agent = create_agent(llm, tools=[get_weather, calculate])

### 1-2. 실행 → 최종 답

질문을 messages로 넣으면 도구 사용을 자동 처리한다.

In [29]:
# TODO: agent.invoke에 messages 형식으로 질문을 넣으세요
msg = {
    "messages": [
        {
            "role": "user",
            "content": "서울 날씨의 현재 온도에 10도가 올라가면 몇 도가 되나요?"
        }
    ]
}

agent_result = agent.invoke(msg)

In [30]:
agent_result['messages'][-1].content

'서울의 현재 온도는 23도입니다. 여기에 10도가 올라가면 33도가 됩니다.'

### 1-3. 실행 과정 들여다보기

도구 호출·결과·최종 답이 messages에 순서대로 담긴다.

In [31]:
# (관찰) 도구 호출 → 도구 결과 → 최종 답이 순서대로 담긴다
for m in agent_result['messages']:
    m.pretty_print()


================================ Human Message =================================

서울 날씨의 현재 온도에 10도가 올라가면 몇 도가 되나요?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_b0PaLlEDQVLL0fJ74owE2sjC)
 Call ID: call_b0PaLlEDQVLL0fJ74owE2sjC
  Args:
    city: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음, 23도
================================== Ai Message ==================================
Tool Calls:
  calculate (call_aDjwcQhp7leUF46UUENxvbtF)
 Call ID: call_aDjwcQhp7leUF46UUENxvbtF
  Args:
    a: 23
    b: 10
    op: +
================================= Tool Message =================================
Name: calculate

33.0
================================== Ai Message ==================================

서울의 현재 온도는 23도입니다. 여기에 10도가 올라가면 33도가 됩니다.


### 1-4. 여러 질문

도구가 필요한 질문·필요 없는 질문을 섞어 본다.

In [54]:
# TODO: 도구가 필요한 질문과 필요 없는 질문을 섞어 리스트로 만드세요 (예: 날씨·계산·인사)
# TODO: 각 질문을 agent.invoke로 실행하고 최종 답만 출력하세요

questions = ["부산 날씨 어때?", "안녕?", "555에 567를 합하면 ?", "999를 0으로 나눠주세요."]

for q in questions:
    result = agent.invoke({"messages":[
        {
            "role": "user",
            "content": q
        }
    ]})
    
    print(result['messages'][-1].content)



부산의 현재 날씨는 맑고, 기온은 23도입니다.
안녕하세요! 어떻게 도와드릴까요?
555에 567을 합하면 1122입니다.
0으로 나누는 것은 수학적으로 정의되지 않기 때문에 수행할 수 없습니다. 다른 계산 요청이 있으면 말씀해 주세요!


### 1-5. 오류 다뤄보기 — 도구 실행 중 예외가 나면?

0으로 나누기를 막지 않은 도구를 하나 추가해, 에이전트가 도구 오류를 어떻게 다루는지 관찰한다.

In [52]:
#도구 정의 : 0으로 나누기를 오류처리 하지 않음
@tool
def divide(a: float, b: float) -> float:
    """a를 b로 나눈다"""
    return a / b


# divide 도구를 가진 agent 호출
user_msg = {"messages":[{"role": "user","content":"999 나누기 0"}]}
agent_div = create_agent(llm, tools=[divide])

try:
    result = agent_div.invoke(user_msg)
    print()
    for m in result['messages']:
        m.pretty_print()
except Exception as e:
    print("오류 종류", type(e).__name__)

result['messages'][-1].content

오류 종류 ZeroDivisionError


'999를 3으로 나누면 333입니다.'

### 1-6. 오류 다뤄보기 — 도구 설명이 모호하면?

M2 Day02의 교훈(설명의 힘)이 에이전트에도 그대로 적용되는지 확인한다.

## Part 2. 다단계 복합 질문

검색과 계산이 함께 필요한 질문을 에이전트가 스스로 나눠 처리하는 과정을 관찰한다.

### 2-1. 검색+계산 도구로 에이전트 재구성

In [ ]:


# TODO: search_population과 calculate를 함께 등록해 agent2를 만드세요




### 2-2. 복합 질문 (검색 후 계산)

In [ ]:
# TODO: "서울 인구를 검색해서 2로 나누면 몇 명이야?"를 agent2에 넣어 result2에 담으세요



### 2-3. 실행 과정 — 도구 여러 번 호출

In [ ]:
# (관찰) 도구가 몇 번, 어떤 순서로 호출됐는지 확인한다


### 2-4. 다른 복합 질문

In [ ]:
# TODO: 검색만 필요한 질문과, 검색 뒤 계산까지 필요한 질문을 하나씩 만드세요
# TODO: 두 질문을 agent2로 실행하고 질문과 최종 답을 함께 출력하세요




## Part 2-확장. 다른 도메인에 적용하기 — M1·M2 도구를 에이전트로

M2 Day01·02에서 만든 채용 요건·회사 리뷰 도구를 에이전트에 맡기면, 복합 질문도 알아서 나눠 처리하는지 확인한다.

### 2-5. 채용 요건+회사 리뷰 에이전트

In [59]:
@tool
def get_job_requirements(company: str, position: str) -> str:
    """회사·직무의 채용 공고 요건을 조회한다."""
    return f"{company}의 {position} 공고 요건: 3년 이상 경력, Python·SQL 우대, 팀 협업 경험 필수"

@tool
def get_company_review(company: str) -> str:
    """회사의 재직자 리뷰 요약을 반환한다. company는 회사 이름(예: 카카오, 라인)."""
    reviews = {
        "카카오": "워라밸이 좋고 자율 출퇴근 문화, 성장 속도는 팀마다 다름",
        "라인": "글로벌 협업 기회가 많고, 일본어 소통이 잦은 편",
    }
    return reviews.get(company, "리뷰 정보가 없습니다")

In [60]:
# TODO: 두 도구를 넘겨 coach_agent_basic을 만드세요
coach_agent_basic = create_agent(llm, tools=[get_job_requirements, get_company_review])

In [62]:
r = coach_agent_basic.invoke({
    "messages": [{"role": "user",
                  "content": "카카오 백엔드 개발자 채용 요건이랑 회사 분위기 둘 다 알려줘"}]
})

In [63]:
print("최종 답:", r["messages"][-1].content)
print()
for m in r['messages']:
    m.pretty_print()
print('='*20)

최종 답: 카카오의 백엔드 개발자 채용 요건은 다음과 같습니다:
- 3년 이상의 경력
- Python 및 SQL 우대
- 팀 협업 경험 필수

회사 분위기는 다음과 같습니다:
- 워라밸(일과 삶의 균형)이 좋고 자율 출퇴근 문화가 형성되어 있음
- 성장 속도는 팀마다 다르게 나타남

더 궁금한 점이 있으면 말씀해 주세요!

================================ Human Message =================================

카카오 백엔드 개발자 채용 요건이랑 회사 분위기 둘 다 알려줘
================================== Ai Message ==================================
Tool Calls:
  get_job_requirements (call_jfWUPxqqS4WnGRVAMMX1LSiV)
 Call ID: call_jfWUPxqqS4WnGRVAMMX1LSiV
  Args:
    company: 카카오
    position: 백엔드 개발자
  get_company_review (call_uWtQ1Q9zVxBAgAJKlPJlfWLt)
 Call ID: call_uWtQ1Q9zVxBAgAJKlPJlfWLt
  Args:
    company: 카카오
================================= Tool Message =================================
Name: get_job_requirements

카카오의 백엔드 개발자 공고 요건: 3년 이상 경력, Python·SQL 우대, 팀 협업 경험 필수
================================= Tool Message =================================
Name: get_company_review

워라밸이 좋고 자율 출퇴근 문화, 성장 속도는 팀마다 다름
================================== Ai Messa

### 2-6. 실행 과정 관찰

In [ ]:
# (관찰) 도구 2개를 에이전트가 어떻게 조합했는지 확인한다


### 관찰 정리

- 인구+계산 도메인과 채용+리뷰 도메인 모두에서, 에이전트가 도구를 몇 번 불렀는가?
- 한 질문에 도구가 2개 필요할 때, 순서를 우리가 정해주지 않아도 되는 이유는 무엇인가?

### 2-7. 도구 4개를 한 에이전트에 모으면?

Part 2와 Part 2-확장에서 쓴 도구 네 개(`search_population`·`calculate`·`get_job_requirements`·`get_company_review`)를 한 에이전트에 모아 등록한다. 서로 다른 두 도메인이 섞인 질문을 한 번에 던져 본다.

In [ ]:
# TODO: 네 도구를 모두 담아 create_agent로 agent_mixed를 만드세요


### 2-8. 실행 과정 확인

In [ ]:
# (관찰) 도구 4개 중 실제로 호출된 것이 무엇인지 확인한다


### 2-9. 관찰 정리

- 도구가 4개로 늘고 서로 다른 두 도메인이 섞여도, 에이전트가 필요한 도구만 순서대로 골라 불렀는가?
- 실제로 어떤 도구가 몇 번 호출됐는가? `calculate`도 호출됐는가, 아니면 모델이 덧셈을 암산으로 처리했는가?

## Part 3. 미니 프로젝트 — 검색·계산·조회 Multi-Tool Agent

서로 다른 도구 3개를 모아, 질문마다 알맞은 도구를 에이전트가 스스로 고르게 한다.

### 3-1. 도구 3종 구성

In [ ]:


# TODO: search_population·calculate·lookup_order를 tools_all에 모으세요

# TODO: tools_all로 agent3을 만드세요




### 3-2. 복합 시나리오 테스트

In [ ]:
# TODO: 아래 세 경우를 각각 확인할 질문을 scenarios에 담으세요
#       ① 검색 → 계산 2단계  ② 조회 1단계  ③ 도구가 필요 없는 인사
scenarios = [
]

# TODO: 각 질문을 agent3으로 실행하고, 질문과 최종 답을 구분선과 함께 출력하세요




## Part 3-확장. M1~M2 통합 프로젝트 — 면접 코치를 에이전트로 업그레이드

M2 Day01·02에서는 `input_guard`→`agent/coach.invoke`→`output_guard`를 **우리가 직접 순서대로** 호출했다. 오늘은 도구 선택·반복 호출을 에이전트에 맡기고, 방어 로직만 앞뒤에 남긴다.

### 방어 로직 (M1 Day04 재사용)

In [ ]:
# TODO: M1 Day04의 위험 문구·금지어 목록과 input_guard·output_guard를 옮겨오세요






### 에이전트 기반 안전한 면접 코치

### 테스트 — 복합 질문·인젝션 시도

In [ ]:

# TODO: 판정을 조작하려는 인젝션 질문을 넣어보세요

### 실패 시나리오 — 존재하지 않는 회사

In [ ]:
# TODO: 존재하지 않는 가상의 회사 이름으로 secure_agent_coach를 호출해보세요



> **참고:** 실제 실행 결과, 이번엔 모델이 `get_job_requirements`를 아예 부르지 않고 "존재하지도 않는 가상회사"라는 표현만 보고 도구 호출 없이 일반 지식으로 답했다. M2 Day01(수동 dispatch 버전)의 같은 질문에서는 도구가 실제로 호출되어 mock 데이터를 그대로 반환했었다 — **같은 질문이라도 모델이 매번 같은 도구 호출 판단을 내린다는 보장은 없다.** 도구 자체(mock)가 회사 존재 여부를 검증하지 않는다는 한계는 여전하지만, 이번 실행에서는 도구가 호출되지 않아 그 한계가 드러나지 않았다.

### M2 모듈 정리

**확인 질문**
- `secure_agent_coach`가 M2 Day01·02의 `coach_with_tools_v2`와 다른 점은 무엇인가?
- 에이전트에게 맡기고 나서, 우리 코드에 남은 책임은 무엇인가?

## 확인 문제

1. 에이전트를 실행하면 도구를 누가 부르는가?
2. `result["messages"]`에는 무엇이 담기는가?
3. 1-5·1-6에서 확인한 것처럼, 도구 실행 오류·모호한 설명은 에이전트에서 각각 어떻게 나타나는가?
4. Part 2와 Part 2-확장에서, 도메인이 바뀌어도 에이전트의 다단계 처리 방식은 같았는가?
5. Part 3-확장의 `secure_agent_coach`에서, 에이전트에게 맡긴 부분과 우리가 직접 남긴 부분은 각각 무엇인가?
6. 도구가 4개로 늘어도(2-7~2-9) 에이전트는 필요한 도구만 정확히 골라 순서대로 불렀는가?
7. 존재하지 않는 회사를 물었을 때, `create_agent` 버전도 M2 Day01의 수동 버전과 같은 한계를 보였는가? 왜인가?
8. Part 3-확장2의 브라우저 도구는 왜 노트북이 아니라 `.py` 스크립트로 실행하는가?

## 정리·회고

오늘 배운 것을 3줄로 정리해 본다.

1. `create_agent`가 대신 처리해 준 것은 무엇이었는가(0-보충의 손으로 짠 루프와 비교)?
2. 도구가 4개로 늘었을 때(2-7~2-9) 무엇을 관찰했는가?
3. 자동화(create_agent)해도 바뀌지 않는 한계는 무엇이었는가?

작성한 요약과 오늘 코드를 커밋한다.